# Scores: three averages, three answers

Calibration averages agree and ranking averages do not. This notebook scores four examples with
three labels, then a larger synthetic multilabel problem, and uses `aggmetrics.scores` to say
*why* the summaries differ.

In [1]:
# Installs the package only where it is missing (for example in Colab).
import importlib.util
if importlib.util.find_spec("aggmetrics") is None:
    %pip install -q aggmetrics

## The running example

In [2]:
import numpy as np
from aggmetrics.scores import (ScoreMatrix, auc, auc_gaps, auc_parts, diagnose_scores,
                               murphy, ndcg, ndcg_parts, pooled_auc, ranking_duality)

Y = np.array([[1, 1, 0], [1, 0, 1], [0, 1, 0], [0, 1, 0]])
P = np.array([[.7, .6, .1], [.6, .1, .4], [.8, .2, .7], [.4, .3, .2]])
sm = ScoreMatrix(Y, P)
print("Brier, micro/macro/exemplar:", [round(sm.brier(a), 6) for a in ("micro", "macro", "exemplar")])
print("nDCG,  micro/macro/exemplar:", [round(ndcg(sm, a), 4) for a in ("micro", "macro", "exemplar")])
print("AUC,   micro/macro/exemplar:", [round(auc(sm, a), 4) for a in ("micro", "macro", "exemplar")])
print("pooled ROC-AUC:", round(pooled_auc(sm), 4))

Brier, micro/macro/exemplar: [0.270833, 0.270833, 0.270833]
nDCG,  micro/macro/exemplar: [0.8348, 0.7394, 0.7827]
AUC,   micro/macro/exemplar: [0.7778, 0.7222, 0.8333]
pooled ROC-AUC: 0.5972


The Brier score has one value, whichever way it is averaged. nDCG and AUC have three. Each gap is a
covariance, and `auc_gaps` and `ndcg_gaps` write it out.

In [3]:
g = auc_gaps(sm)
print("micro - macro AUC      :", round(g["micro_minus_macro"], 4))
print("Cov(n+, AUC_j)/mean n+ :", round(g["class_covariance"] / g["mean_positives"], 4))
print("contribution per label :", np.round(g["class_contributions"], 4))

micro - macro AUC      : 0.0556
Cov(n+, AUC_j)/mean n+ : 0.0556
contribution per label : [-0.0741  0.1389 -0.0093]


## Two ranking questions

Per-class nDCG asks where a label sits among the labels of its positive examples; AUC asks
whether its positives outscore its negatives. They can order the labels in opposite ways.

In [4]:
d = ranking_duality(sm)
print("nDCG_j:", np.round(d["ndcg"], 3))
print("AUC_j :", np.round(d["auc"], 3))
print("label pairs ordered oppositely:", d["discordant"], " tau =", d["tau"])

nDCG_j: [1.    0.587 0.631]
AUC_j : [0.5   1.    0.667]
label pairs ordered oppositely: [(0, 1), (0, 2), (1, 2)]  tau = -1.0


## When the Murphy decomposition has nothing to say

With distinct scores every level set is one cell, so reliability equals the Brier score. The
default warns; bins or the isotonic (CORP) method give an informative split.

In [5]:
import warnings
rng = np.random.default_rng(0)
n, m = 2000, 4
latent = rng.normal(size=(n, m))
prob = 1 / (1 + np.exp(-latent))
Yb = (rng.random((n, m)) < prob).astype(int)
over = 1 / (1 + np.exp(-2.5 * latent))          # an over-confident model
big = ScoreMatrix(Yb, over)

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    raw = murphy(big)
print("unbinned warns:", bool(w), " REL == BS:", np.allclose(raw.rel, raw.brier))
for kw in ({"bins": 10}, {"method": "isotonic"}):
    r = murphy(big, **kw)
    print(kw, "REL", np.round(r.rel, 4), "RES", np.round(r.res, 4))

unbinned warns: True  REL == BS: True
{'bins': 10} REL [0.0187 0.0213 0.023  0.0192] RES [0.0481 0.0465 0.0416 0.0452]
{'method': 'isotonic'} REL [0.0217 0.0228 0.0256 0.0232] RES [0.0518 0.0496 0.0452 0.0499]


## What the diagnostics report

In [6]:
for f in diagnose_scores(big):
    print(f)

[warning] degenerate_murphy: every score is distinct within its column, so the unbinned Murphy decomposition gives REL = BS and RES = UNC; use murphy(..., bins=k) or method='isotonic'
[note] ndcg_no_collapse: micro-nDCG 0.9137, class mean 0.7127 and exemplar mean 0.8947 differ; micro - exemplar is Cov(IDCG_i, nDCG_i)/mean IDCG = +0.0190, and class 1 contributes most to micro - class
[note] class_coupling: 1358 examples carry several labels, so replicating the positives of one label also replicates positives of others and moves their alpha, AUC and nDCG; brier_under_prior and the skew-invariance of alpha_j, beta_j, AUC_j and nDCG_j hold class by class


## Replicating positives

`skew_report` repeats the positive examples of each label and records what moves.

In [7]:
from aggmetrics.scores import skew_report
r = skew_report(big, factor=3)
for s in ("alpha", "beta", "auc", "ndcg", "brier"):
    print(f"{s:6s} own-label change:", np.round(r['own'][s], 4))

alpha  own-label change: [0. 0. 0. 0.]
beta   own-label change: [0. 0. 0. 0.]
auc    own-label change: [0. 0. 0. 0.]
ndcg   own-label change: [0. 0. 0. 0.]
brier  own-label change: [0.0012 0.0013 0.003  0.0011]
